<a href="https://colab.research.google.com/github/c-marq/cap4767-data-mining/blob/main/exercises/m04-05_group_exercise_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modules 4–5 Group Exercise, Part 2: Churn, Logistic Regression vs Neural Network
**CAP4767 Data Mining with Python** | Miami Dade College, Kendall Campus

**Points:** 5 (Part 2 of 2; the whole exercise is 10) | **Duration:** ~45 minutes | **Deliverable:** Completed notebook + 2–3 minute presentation

**Submission:** each student submits their own copy in Canvas. The work is done in groups, but every student turns in their own notebook (file name: `LastName_FirstName_Exercise04_Part2.ipynb`).

**Objective:** Build and compare a logistic regression model and a Keras neural network on the Bank Customer Churn dataset (new data, not the Telco data from the demo). Present your confusion matrices, ROC curves, and model recommendation. The steps are the same as the Modules 4–5 demo, but the data is different.

**Companion page:** the Modules 4–5 practice worksheet walks through these same steps on the same bank data, with hints and solutions. Steps A4 and A6 to A10 of the page help with this part. https://c-marq.github.io/cap4767-data-mining/practice/m04-05-churn/

### Your dataset

**Bank Customer Churn** (`Churn_Modelling.csv`): 10,000 customers of a retail bank in France, Germany and Spain. The target `Exited` is 1 if the customer closed their account and 0 if they stayed. About 20% of the customers left. Source: Kaggle, Churn Modelling (shrutimechlearn). This is new data, not the Telco data from the demo.

### Group Members & Roles

| Role | Name | Responsibility |
|------|------|----------------|
| 🖥️ **Lead Coder** | | Drives the notebook |
| 📊 **Data Interpreter** | | Reads outputs, explains metrics |
| 🎤 **Presenter** | | Delivers the 2–3 minute share-out |
| ✅ **QA Reviewer** | | Checks outputs against checkpoints |

*If 3 members, Lead Coder also handles QA.*

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 GROUP DISCUSSION (before coding, 3 minutes)</strong><br>
  A bank has a retention offer worth $500 per customer (a fee waiver and a rate bonus). It can only afford to make the offer to 200 customers this quarter, but the churn model flags 350 as high-risk.
  <ol>
    <li>What happens if they target the wrong 200?</li>
    <li>Would you rather have a model with high <strong>precision</strong> (fewer false alarms) or high <strong>recall</strong> (catches more churners)? Why?</li>
    <li>Is there a scenario where the "worse" model on paper is the better business choice?</li>
  </ol>
</div>

**Our group's answers (minimum 3 sentences):**

*(Write here)*

---

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Run the setup cell below. It loads the dataset, runs the full preprocessing pipeline (the same steps as the demo), and creates the train/test split. <strong>Do not modify.</strong>
</div>

In [ ]:
# ============================================================
# Setup. Run this cell. Do not modify.
# Full preprocessing pipeline, same steps as the demo.
# ============================================================
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, roc_auc_score,
                             accuracy_score, precision_score, recall_score, f1_score)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

# Load + preprocess
url = "https://raw.githubusercontent.com/c-marq/cap4767-data-mining/refs/heads/main/data/Churn_Modelling.csv"
df = pd.read_csv(url)
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])  # row number, ID and name are not predictors

df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})
# Geography and NumOfProducts become 0/1 dummy columns. NumOfProducts is a category on purpose:
# 2 products churn least and 3 or 4 churn most, which one straight-line coefficient cannot show.
df = pd.get_dummies(df, columns=["Geography", "NumOfProducts"],
                     drop_first=True, dtype=int)

X = df.drop(columns=["Exited"])
y = df["Exited"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
continuous = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]
X_train[continuous] = scaler.fit_transform(X_train[continuous])
X_test[continuous] = scaler.transform(X_test[continuous])

feature_names = X_train.columns.tolist()
n_features = len(feature_names)

print(f"✅ Preprocessing complete")
print(f"   Features: {n_features} | Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"   Churn rate, Train: {y_train.mean():.1%} | Test: {y_test.mean():.1%}")

# Note: Results may vary slightly across runs even with seeds set.
print(f"   TensorFlow: {tf.__version__}")

---
## Task 1: Build the Logistic Regression Model (0.5 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Build and train a <code>LogisticRegression</code> on the scaled training data. Use <code>max_iter=1000, random_state=42</code>.<br>
  Store predictions in <code>lr_predictions</code> and probabilities in <code>lr_probabilities</code>.
</div>

In [ ]:
# Task 1: Logistic Regression
# YOUR CODE HERE
# Store: lr_predictions, lr_probabilities


---
## Task 2: Classification Report + Interpretation (0.5 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Print the classification report using <code>target_names=['Stayed', 'Churned']</code>.
</div>

In [ ]:
# Task 2: Classification report
# YOUR CODE HERE


**Interpretation (2–3 sentences):** What do precision and recall mean *specifically for the "Churned" class*? Which metric matters more for a retention team, and why?

*(Write here)*

<div style="background-color: #FADBD8; border-left: 5px solid #E74C3C; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #922B21;">🛑 CHECKPOINT 1</strong><br>
  LR should show ≈84% accuracy and 30–45% recall on Churned (about 36%). If recall is below 25% or above 55%, check preprocessing.
</div>

---
## Task 3: Build the Keras Neural Network (1 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Build a Keras Sequential model:
  <ul>
    <li>Hidden layer 1: <code>n_features</code> neurons, ReLU</li>
    <li>Dropout: 0.3</li>
    <li>Hidden layer 2: 15 neurons, ReLU</li>
    <li>Dropout: 0.2</li>
    <li>Output: 1 neuron, sigmoid</li>
  </ul>
  Compile with Adam, binary crossentropy and <code>metrics=['accuracy']</code> (Task 5 plots accuracy). Print <code>model.summary()</code>.
</div>

In [ ]:
# Task 3: Build ANN
# YOUR CODE HERE


---
## Task 4: Train with Early Stopping (0.5 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Train with <code>epochs=200, batch_size=32, validation_split=0.2</code>.<br>
  Use <code>EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)</code>.<br>
  Store the result of <code>fit</code> as <code>history</code> and print how many epochs ran: <code>len(history.history['loss'])</code>.
</div>

In [ ]:
# Task 4: Train with early stopping
# YOUR CODE HERE


---
## Task 5: Plot Training Curves (0.5 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Side-by-side plot (1 row, 2 cols): training vs validation loss (left) and accuracy (right).
</div>

In [ ]:
# Task 5: Training curves
# YOUR CODE HERE


**Interpretation (2–3 sentences):** Is there evidence of overfitting? How can you tell from the curves?

*(Write here)*

<div style="background-color: #FADBD8; border-left: 5px solid #E74C3C; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #922B21;">🛑 CHECKPOINT 2</strong><br>
  Training should stop somewhere between about epoch 35 and 130 (most runs: 45 to 85; it changes every run, even with the seeds set). Validation loss should track close to training loss and can sit a little below it, because dropout is only on while training. If it ran all 200 epochs, check the EarlyStopping config (the <code>callbacks</code> list) and that <code>fit</code> has <code>validation_split=0.2</code>.
</div>

---
## Task 6: Evaluate the ANN (0.5 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Generate predictions (threshold=0.5) and probabilities. Print classification report.<br>
  Store in <code>ann_predictions</code> and <code>ann_probabilities</code>.
</div>

In [ ]:
# Task 6: Evaluate ANN
# YOUR CODE HERE
# Store: ann_predictions, ann_probabilities


---
## Task 7: ROC Curve Comparison (1 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Plot both ROC curves on a single figure. LR = navy <code>#0f3460</code>, ANN = coral <code>#e94560</code>. Show AUC in legend.
</div>

In [ ]:
# Task 7: ROC curve comparison
# YOUR CODE HERE


<div style="background-color: #FADBD8; border-left: 5px solid #E74C3C; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #922B21;">🛑 CHECKPOINT 3</strong><br>
  The ANN should edge out logistic regression on this dataset: AUC about 0.85 to 0.87 for the ANN against about 0.84 for LR, ANN recall roughly 0.40 to 0.52 (LR is about 36%), and ANN accuracy about 0.86 (LR is about 0.84). The gap is small, and it changes a little every run. If your ANN comes out slightly behind LR, that is still a fine result to write up. If the ANN's recall is under about 38% (a correct network rarely falls that low), its accuracy is stuck near 0.80, or its recall is over 90%, check the last layer: run <code>print(model.layers[-1].get_config()['activation'])</code> (use your own variable name if the model is not called <code>model</code>). It must say <code>sigmoid</code>. If it does, the output layer is right.
</div>

---
## Task 8: Model Recommendation (0.5 pt)

**Answer all of the following (minimum 4 sentences):**

1. Which model has better recall on churners?
2. Which model has better AUC?
3. Which model can explain *why* a customer is flagged?
4. If the company can only pick one model, which one and why?
5. Is there a scenario where deploying both makes sense?

*(Write here)*

---

## Troubleshooting

| Problem | Fix |
|---------|-----|
| ANN accuracy is about 0.80 and doesn't change | Model is predicting all "Stayed": check architecture and compilation |
| ANN accuracy is about 0.20 and recall = 1.00 | Every customer is predicted "Churned". The last layer must be `Dense(1, activation="sigmoid")`, not `softmax` |
| ANN recall under about 38% while accuracy is still about 0.85 | Check the last layer: `print(model.layers[-1].get_config()['activation'])` (use your own model's variable name) must say `sigmoid` (a `relu` or a missing activation does this) |
| `ValueError: ... expected axis -1 of input shape to have value 5, but received input with shape (32, 13)` | Check that `input_shape=(n_features,)` matches your data (this dataset has 13 features) |
| Training runs all 200 epochs | EarlyStopping is not in the `callbacks` list, or `validation_split=0.2` is missing from `fit` (EarlyStopping needs `val_loss` to watch). Check `model.fit(..., validation_split=0.2, callbacks=[early_stop])` |
| ROC curve is a bent line with a low AUC (about 0.66 to 0.71) | You're plotting predictions (0/1) instead of probabilities: use `predict_proba` or `model.predict` |

---
<p style="color:#7F8C8D; font-size:0.85em;">
<em>CAP4767 Data Mining with Python | Miami Dade College</em><br>
Modules 4–5 Group Exercise, Part 2: Churn, LR vs ANN | 5 Points
</p>